In [1]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os
from pathlib import Path

import dask
import dask.array as da
from dask.distributed import LocalCluster, Client
from datetime import datetime

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
workingDir = Path().absolute()
print(f"{workingDir}")

/g/data/ng72/ms5578/ID_HW_BARRA


In [3]:
client = Client()
client

2025-04-14 12:08:44,378 - distributed.preloading - INFO - Creating preload: /g/data/hh5/public/apps/dask-optimiser/schedplugin.py
2025-04-14 12:08:44,380 - distributed.utils - INFO - Reload module schedplugin from .py file
2025-04-14 12:08:44,386 - distributed.preloading - INFO - Import preload module: /g/data/hh5/public/apps/dask-optimiser/schedplugin.py


Modifying workers


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /node/gadi-cpu-bdw-0312.gadi.nci.org.au/7986/proxy/8787/status,
Dashboard: /node/gadi-cpu-bdw-0312.gadi.nci.org.au/7986/proxy/8787/status,Workers: 28
Total threads: 28,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37341,Workers: 28
Dashboard: /node/gadi-cpu-bdw-0312.gadi.nci.org.au/7986/proxy/8787/status,Total threads: 28
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:32989,Total threads: 1
Dashboard: /node/gadi-cpu-bdw-0312.gadi.nci.org.au/7986/proxy/42839/status,Memory: 0 B
Nanny: tcp://127.0.0.1:34995,


2025-04-14 12:09:46,241 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 4128c4b5bb542c59f93f22deed884fd6 initialized by task ('rechunk-merge-rechunk-transfer-45533b194a24d6d67b95d15f2437dc0d', 0, 4, 6, 12, 7, 6) executed on worker tcp://127.0.0.1:39755
2025-04-14 12:09:47,574 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle b76fc091f67aaeeb17546b18afb0633f initialized by task ('rechunk-merge-rechunk-transfer-45533b194a24d6d67b95d15f2437dc0d', 0, 0, 3, 43, 0, 3) executed on worker tcp://127.0.0.1:36791
2025-04-14 12:09:47,684 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2735c5927d7e34e898d1d17ba68b0aae initialized by task ('rechunk-merge-rechunk-transfer-45533b194a24d6d67b95d15f2437dc0d', 0, 0, 5, 43, 0, 5) executed on worker tcp://127.0.0.1:37263
2025-04-14 12:09:47,935 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 79d35d2878336ac877229735e9b5d39c initialized by task ('rechunk-merge-rechunk-transfer-45533b194a24d6d67b95d15f2437dc0d', 

In [4]:
tas_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/tas/latest/"
write_path = f'{workingDir}/data/preprocess/'

In [5]:
sdate, edate ='19790101', '20001231'

In [6]:
fdates = [m.strftime('%Y%m') for m in pd.date_range(sdate, edate, freq='ME')]
fnames = [s for s in os.listdir(tas_path) if any(f in s for f in fdates)]
fpaths = sorted([tas_path + f for f in fnames])

tas_ds = xr.open_mfdataset(fpaths, concat_dim ='time', combine='nested', 
                           parallel=True, data_vars='minimal',coords='minimal', 
                           drop_variables = "time_bnds",chunks="auto")
datestr = f"s{sdate}_e{edate}"

t95 = tas_ds.reduce(np.nanpercentile, q=95, dim="time")
t95 = t95.rename(name_dict={'tas':'PRCTILE95'})

In [ ]:
encoding = {"PRCTILE95":{"zlib": True, "complevel": 4, "shuffle": True}}

write_task = t95.to_netcdf(f'{write_path}t95_baseline.nc',
                                        encoding=encoding,
                                        compute=False,
                                        engine="netcdf4")
dask.compute(write_task)

/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.07/lib/python3.10/site-packages/distributed/client.py:3371: UserWarning: Sending large graph of size 26.10 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
